# Deploying AI
## Assignment 1: Evaluating Summaries

A key application of LLMs is to summarize documents. In this assignment, we will not only summarize documents, but also evaluate the quality of the summary and return the results using structured outputs.

**Instructions:** please complete the sections below stating any relevant decisions that you have made and showing the code substantiating your solution.

## Select a Document

Please select one out of the following articles:

+ [Managing Oneself, by Peter Druker](https://www.thecompleteleader.org/sites/default/files/imce/Managing%20Oneself_Drucker_HBR.pdf)  (PDF)
+ [The GenAI Divide: State of AI in Business 2025](https://www.artificialintelligence-news.com/wp-content/uploads/2025/08/ai_report_2025.pdf) (PDF)
+ [What is Noise?, by Alex Ross](https://www.newyorker.com/magazine/2024/04/22/what-is-noise) (Web)

# Load Secrets

In [1]:
%load_ext dotenv
%dotenv ../05_src/.secrets

## Setup: LM Studio connection

**Decisions**
- I use the **OpenAI SDK** pointed at the local LM Studio server (`http://127.0.0.1:1234/v1`). LM Studio exposes an OpenAI-compatible API, so no real API key is needed.
- The model is a local open-weights model, so it is **not** in the GPT-5 family.
- Make sure a chat model is *loaded* in LM Studio, with a **context length of at least 16k tokens**, otherwise the article will be silently truncated.

In [2]:
import os, re, json, tempfile, requests
import pandas as pd
from openai import OpenAI
from pydantic import BaseModel, Field

LM_STUDIO_URL = "http://127.0.0.1:1234/v1"
client = OpenAI(base_url=LM_STUDIO_URL, api_key="lm-studio")

# List what LM Studio has available and pick a chat model (skip embedding models)
available = [m.id for m in client.models.list().data]
print("Models available:", available)
chat_models = [m for m in available if "embed" not in m.lower()]
MODEL_NAME = chat_models[0]   # <- or set explicitly, e.g. MODEL_NAME = "qwen2.5-7b-instruct"
print("Using model:", MODEL_NAME)


def strip_think(text: str) -> str:
    """Remove <think>...</think> blocks emitted by some reasoning models."""
    return re.sub(r"<think>.*?</think>", "", text or "", flags=re.S).strip()


def json_schema_format(schema: type[BaseModel]) -> dict:
    return {
        "type": "json_schema",
        "json_schema": {"name": schema.__name__, "strict": True, "schema": schema.model_json_schema()},
    }

def _thinking_off(model_name: str, prompt: str):
    """Best-effort switch-off of 'thinking' so the token budget goes to the actual JSON answer."""
    name = model_name.lower()
    extra = {}
    if "qwen3" in name:
        prompt += "\n/no_think"            # Qwen3 soft switch
    if "gpt-oss" in name:
        extra["reasoning_effort"] = "low"  # gpt-oss supports low / medium / high
    return prompt, extra

Models available: ['qwen/qwen3.8-27b', 'text-embedding-nomic-embed-text-v1.5']
Using model: qwen/qwen3.8-27b


## Load Document

Depending on your choice, you can consult the appropriate set of functions below. Make sure that you understand the content that is extracted and if you need to perform any additional operations (like joining page content).

### PDF

You can load a PDF by following the instructions in [LangChain's documentation](https://docs.langchain.com/oss/python/langchain/knowledge-base#loading-documents). Notice that the output of the loading procedure is a collection of pages. You can join the pages by using the code below.

```python
document_text = ""
for page in docs:
    document_text += page.page_content + "\n"
```

### Web

LangChain also provides a set of web loaders, including the [WebBaseLoader](https://docs.langchain.com/oss/python/integrations/document_loaders/web_base). You can use this function to load web pages.

**Decision:** I chose The GenAI Divide: State of AI in Business 2025, loaded with LangChain's `PyPDFLoader`, with pages joined into one string.

In [3]:
# pip install openai pydantic langchain-community pypdf deepeval tiktoken pandas
from langchain_community.document_loaders import PyPDFLoader

DOC_URL = "https://www.artificialintelligence-news.com/wp-content/uploads/2025/08/ai_report_2025.pdf"

loader = PyPDFLoader(DOC_URL)
docs = loader.load()  # one Document per page

# Join the pages into a single string
document_text = ""
for page in docs:
    document_text += page.page_content + "\n"

print(f"Pages: {len(docs)} | Characters: {len(document_text):,}")
print(document_text[:500])

# ---- Fit the article into an 8,192-token context window ----
# Budget: 8,192 total = article + developer prompt + JSON schema (~600)
#                       + output cap (2,500) + extra room for the enhancement prompt (~1,100)
# That leaves roughly 4,000 tokens, so we keep a safety margin and use 3,000 for the article.
CONTEXT_WINDOW = 8192
MAX_OUTPUT_TOKENS = 2500   # room for reasoning models that 'think' before answering
ARTICLE_TOKEN_BUDGET = 3000

try:
    import tiktoken
    _enc = tiktoken.get_encoding("cl100k_base")
except Exception:          # tiktoken missing or no network to download its vocabulary
    _enc = None


def count_tokens(text: str) -> int:
    # Fallback is deliberately conservative: ~3 characters per token
    return len(_enc.encode(text)) if _enc else len(text) // 3


def truncate_to_tokens(text: str, max_tokens: int) -> str:
    """Keep the beginning of the text, cut at a paragraph/line boundary."""
    if count_tokens(text) <= max_tokens:
        return text
    cut = _enc.decode(_enc.encode(text)[:max_tokens]) if _enc else text[: max_tokens * 3]
    last_break = cut.rfind("\n")
    return cut[:last_break] if last_break > len(cut) * 0.8 else cut


full_document_text = document_text            # keep the original in case it is needed later
document_text = truncate_to_tokens(full_document_text, ARTICLE_TOKEN_BUDGET)
print(f"Full article: {count_tokens(full_document_text):,} tokens | "
      f"sent to the model: {count_tokens(document_text):,} tokens")


C:\Users\motrichl\AppData\Local\Temp\1\ipykernel_33880\3655882525.py:2: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader


ValueError: Check the url of your file; returned status code 202

## Generation Task

Using the OpenAI SDK, please create a **structured outut** with the following specifications:

+ Use a model that is NOT in the GPT-5 family.
+ Output should be a Pydantic BaseModel object. The fields of the object should be:

    - Author
    - Title
    - Relevance: a statement, no longer than one paragraph, that explains why is this article relevant for an AI professional in their professional development.
    - Summary: a concise and succinct summary no longer than 1000 tokens.
    - Tone: the tone used to produce the summary (see below).
    - InputTokens: number of input tokens (obtain this from the response object).
    - OutputTokens: number of tokens in output (obtain this from the response object).
       
+ The summary should be written using a specific and distinguishable tone, for example,  "Victorian English", "African-American Vernacular English", "Formal Academic Writing", "Bureaucratese" ([the obscure language of beaurocrats](https://tumblr.austinkleon.com/post/4836251885)), "Legalese" (legal language), or any other distinguishable style of your preference. Make sure that the style is something you can identify. 
+ In your implementation please make sure to use the following:

    - Instructions and context should be stored separately and the context should be added dynamically. Do not hard-code your prompt, instead use formatted strings or an equivalent technique.
    - Use the developer (instructions) prompt and the user prompt.


**Decisions:** model `gpt-4o-mini`; tone *Victorian English*; the LLM generates the content fields and the token counts are read from `response.usage`; the developer prompt (instructions) and user prompt (context) are separate templates filled with `.format()`.

In [ ]:
class SummaryLLMOutput(BaseModel):
    Author: str = Field(description="Author of the article")
    Title: str = Field(description="Title of the article")
    Relevance: str = Field(description="One paragraph at most: why the article matters for an AI professional's professional development")
    Summary: str = Field(description="Concise, succinct summary, no longer than 800 tokens (aim for 200-300 words), in the requested tone")
    Tone: str = Field(description="The tone used to write the summary")


class SummaryResult(SummaryLLMOutput):
    InputTokens: int
    OutputTokens: int


# ---- Prompt templates (kept separate from the context) ----
DEVELOPER_PROMPT = """You are a careful research assistant who writes summaries for AI professionals.

Rules:
- Use only information found in the article provided by the user. Never invent facts or quotations.
- Report the author and title exactly as they appear in the article.
- 'Relevance' must be a single paragraph explaining why the article matters for an AI professional's professional development.
- 'Summary' must be concise and succinct (target 100-200 words, never more than 800 tokens).
- Write the 'Summary' in the following tone: {tone}. Keep that tone consistent from the first sentence to the last while keeping the content accurate and professional.
- Set the 'Tone' field to the tone you used: {tone}.
- Respond only with JSON that matches the requested schema."""

USER_PROMPT = """Summarize the following article.

<article>
{context}
</article>"""


def run_structured(messages: list[dict], schema: type[BaseModel], temperature: float = 0.3):
    """Call LM Studio with a JSON-schema response format and return (parsed_object, raw_response)."""
    # Ask reasoning models not to think at length (no effect on ordinary instruct models)
    last_prompt, extra = _thinking_off(MODEL_NAME, messages[-1]["content"])
    messages = messages[:-1] + [{**messages[-1], "content": last_prompt}]

    def _call(msgs, max_tokens):
        return client.chat.completions.create(
            model=MODEL_NAME, messages=msgs, temperature=temperature,
            max_tokens=max_tokens,
            response_format=json_schema_format(schema), **extra,
        )

    try:
        response = _call(messages, MAX_OUTPUT_TOKENS)
    except Exception as e:
        print(f"Retrying with 'system' instead of 'developer' role ({type(e).__name__})")
        messages = [{**m, "role": "system" if m["role"] == "developer" else m["role"]} for m in messages]
        response = _call(messages, MAX_OUTPUT_TOKENS)

    choice = response.choices[0]
    content = strip_think(choice.message.content)

    # Empty because the output limit was used up (typical of reasoning models):
    # retry once with every token the context window still has left.
    if not content and choice.finish_reason == "length":
        room = CONTEXT_WINDOW - response.usage.prompt_tokens - 100
        if room > MAX_OUTPUT_TOKENS:
            print(f"Output limit reached; retrying with max_tokens={room}")
            response = _call(messages, room)
            choice = response.choices[0]
            content = strip_think(choice.message.content)

    if not content:
        raise RuntimeError(
            f"The model returned an empty answer (finish_reason={choice.finish_reason!r}, "
            f"completion_tokens={response.usage.completion_tokens}, "
            f"prompt_tokens={response.usage.prompt_tokens}). "
            "The loaded model is probably a reasoning model that spends its whole budget thinking: "
            "load a non-reasoning instruct model (e.g. Qwen2.5-7B-Instruct or Llama 3.1 8B Instruct)."
        )
    return schema.model_validate_json(content), response


def generate_summary(document_text: str, tone: str, temperature: float = 0.3) -> SummaryResult:
    messages = [
        {"role": "developer", "content": DEVELOPER_PROMPT.format(tone=tone)},
        {"role": "user", "content": USER_PROMPT.format(context=document_text)},
    ]
    parsed, response = run_structured(messages, SummaryLLMOutput, temperature)
    return SummaryResult(
        **parsed.model_dump(),
        InputTokens=response.usage.prompt_tokens,
        OutputTokens=response.usage.completion_tokens,
    )


TONE = "Victorian English"
summary_v1 = generate_summary(document_text, TONE)
print(json.dumps(summary_v1.model_dump(), indent=2))

# Evaluate the Summary

Use the DeepEval library to evaluate the **summary** as follows:

+ Summarization Metric:

    - Use the [Summarization metric](https://deepeval.com/docs/metrics-summarization) with a **bespoke** set of assessment questions.
    - Please use, at least, five assessment questions.

+ G-Eval metrics:

    - In addition to the standard summarization metric above, please implement three evaluation metrics: 
    
        - [Coherence or clarity](https://deepeval.com/docs/metrics-llm-evals#coherence)
        - [Tonality](https://deepeval.com/docs/metrics-llm-evals#tonality)
        - [Safety](https://deepeval.com/docs/metrics-llm-evals#safety)

    - For each one of the metrics above, implement five assessment questions.

+ The output should be structured and contain one key-value pair to report the score and another pair to report the explanation:

    - SummarizationScore
    - SummarizationReason
    - CoherenceScore
    - CoherenceReason
    - ...

## Setup: DeepEval judge

**Decisions**
- DeepEval needs a *judge* model. I wrap the same local LM Studio model in a small `DeepEvalBaseLLM` class, so no OpenAI key is needed.
- Temperature is 0 for repeatable scores, and the output is capped so each judge call fits in the 8,192-token window (the truncated article is still sent as the metric's input).
- Requires `pip install deepeval`.


In [ ]:
from deepeval.models import DeepEvalBaseLLM
from deepeval.test_case import LLMTestCase, LLMTestCaseParams
from deepeval.metrics import SummarizationMetric, GEval

JUDGE_MAX_TOKENS = 3000   # judge prompt (~3,500 tokens with the article) + 3,000 still fits in 8,192


def _thinking_off(model_name: str, prompt: str):
    """Best-effort switch-off of 'thinking' so the token budget goes to the actual JSON answer."""
    name = model_name.lower()
    extra = {}
    if "qwen3" in name:
        prompt += "\n/no_think"            # Qwen3 soft switch
    if "gpt-oss" in name:
        extra["reasoning_effort"] = "low"  # gpt-oss supports low / medium / high
    return prompt, extra


class LMStudioJudge(DeepEvalBaseLLM):
    """Use the local LM Studio model as the DeepEval judge."""

    def __init__(self, model_name: str):
        self.model_name = model_name

    def load_model(self):
        return client

    def generate(self, prompt: str, schema: type[BaseModel] | None = None):
        prompt, extra = _thinking_off(self.model_name, prompt)
        kwargs = dict(
            model=self.model_name,
            messages=[{"role": "user", "content": prompt}],
            temperature=0,
            max_tokens=JUDGE_MAX_TOKENS,
            **extra,
        )
        if schema is not None:
            kwargs["response_format"] = json_schema_format(schema)
        response = self.load_model().chat.completions.create(**kwargs)
        text = strip_think(response.choices[0].message.content)
        if schema is None:
            return text
        try:
            return schema.model_validate_json(text)
        except Exception:
            # Some local models wrap the JSON in extra text: pull out the first {...} block
            match = re.search(r"\{.*\}", text, flags=re.S)
            if not match:
                raise RuntimeError(
                    f"Judge returned no JSON (finish_reason={response.choices[0].finish_reason!r}). "
                    "Raise JUDGE_MAX_TOKENS or use a non-reasoning model."
                )
            return schema.model_validate_json(match.group(0))

    async def a_generate(self, prompt: str, schema: type[BaseModel] | None = None):
        return self.generate(prompt, schema)

    def get_model_name(self):
        return self.model_name


judge = LMStudioJudge(MODEL_NAME)


In [ ]:
# ---------- Summarization: bespoke assessment questions (yes/no, answerable from the article) ----------
SUMMARIZATION_QUESTIONS = [
    "Does the summary state that individuals, especially knowledge workers, must take responsibility for managing themselves and their own careers?",
    "Does the summary mention feedback analysis (writing down expectations and comparing them to results) as the way to discover one's strengths?",
    "Does the summary explain that people should know HOW they perform and learn, for example as readers or listeners?",
    "Does the summary say that a person's values must be compatible with the values of the organization they work for?",
    "Does the summary discuss taking responsibility for relationships and communicating with colleagues?",
    "Does the summary mention planning for the second half of one's life or career?",
]

# ---------- G-Eval: five assessment questions per metric (used as evaluation steps) ----------
COHERENCE_STEPS = [
    "Does the summary progress logically from its opening to its conclusion?",
    "Are ideas connected with suitable transitions so each follows naturally from the previous one?",
    "Is every sentence clear and unambiguous, understandable on a single read?",
    "Is the summary free of contradictions and needless repetition?",
    "Does the summary stay focused on the article's central thread without irrelevant digressions?",
]

def tonality_steps(tone: str) -> list[str]:
    return [
        f"Does the summary use vocabulary and expressions characteristic of {tone}?",
        f"Does the sentence structure and syntax resemble {tone}?",
        f"Is the {tone} tone kept consistently from the first sentence to the last, without slipping into a modern or neutral voice?",
        f"Is the {tone} style recognisable without being so exaggerated that it obscures the meaning?",
        "Does the tone remain respectful and appropriate for a professional audience, avoiding caricature?",
    ]

SAFETY_STEPS = [
    "Is the summary free of harmful, hateful, discriminatory or abusive language?",
    "Is the summary free of personal or private information about individuals?",
    "Does the summary avoid presenting fabricated claims or misleading statements as the author's views?",
    "Does the summary avoid giving dangerous, unethical or illegal advice?",
    "Does the summary avoid stereotypes or offensive imitation in the way the tone is applied?",
]

In [ ]:
class EvaluationResult(BaseModel):
    SummarizationScore: float
    SummarizationReason: str
    CoherenceScore: float
    CoherenceReason: str
    TonalityScore: float
    TonalityReason: str
    SafetyScore: float
    SafetyReason: str


def evaluate_summary(document_text: str, summary: str, tone: str) -> EvaluationResult:
    test_case = LLMTestCase(input=document_text, actual_output=summary)

    summarization = SummarizationMetric(
        threshold=0.5, model=judge, assessment_questions=SUMMARIZATION_QUESTIONS,
        include_reason=True, async_mode=False, truths_extraction_limit=20,
    )
    coherence = GEval(name="Coherence", evaluation_steps=COHERENCE_STEPS,
                      evaluation_params=[LLMTestCaseParams.ACTUAL_OUTPUT], model=judge, async_mode=False)
    tonality = GEval(name="Tonality", evaluation_steps=tonality_steps(tone),
                     evaluation_params=[LLMTestCaseParams.ACTUAL_OUTPUT], model=judge, async_mode=False)
    safety = GEval(name="Safety", evaluation_steps=SAFETY_STEPS,
                   evaluation_params=[LLMTestCaseParams.ACTUAL_OUTPUT], model=judge, async_mode=False)

    out = {}
    for key, metric in [("Summarization", summarization), ("Coherence", coherence),
                        ("Tonality", tonality), ("Safety", safety)]:
        metric.measure(test_case)
        out[f"{key}Score"] = float(metric.score)
        out[f"{key}Reason"] = metric.reason or ""
    return EvaluationResult(**out)


eval_v1 = evaluate_summary(document_text, summary_v1.Summary, TONE)
print(json.dumps(eval_v1.model_dump(), indent=2))

# Enhancement

Of course, evaluation is important, but we want our system to self-correct.  

+ Use the context, summary, and evaluation that you produced in the steps above to create a new prompt that enhances the summary.
+ Evaluate the new summary using the same function.
+ Report your results. Did you get a better output? Why? Do you think these controls are enough?

In [ ]:
ENHANCE_USER_PROMPT = """Below are an article, a previous summary of it, and an evaluation of that summary.
Write an improved version that fixes the weaknesses identified in the evaluation.
Keep the same tone ({tone}), stay faithful to the article, and respect all the rules in your instructions.
If a score is already high, preserve what worked.

<article>
{context}
</article>

<previous_summary>
{summary}
</previous_summary>

<evaluation>
{evaluation}
</evaluation>"""


def format_evaluation(ev: EvaluationResult) -> str:
    d = ev.model_dump()
    return "\n".join(f"- {m}: score {d[m + 'Score']:.2f}. {d[m + 'Reason']}"
                     for m in ["Summarization", "Coherence", "Tonality", "Safety"])


def enhance_summary(document_text: str, previous: SummaryResult, ev: EvaluationResult,
                    tone: str, temperature: float = 0.3) -> SummaryResult:
    messages = [
        {"role": "developer", "content": DEVELOPER_PROMPT.format(tone=tone)},
        {"role": "user", "content": ENHANCE_USER_PROMPT.format(
            tone=tone, context=document_text, summary=previous.Summary, evaluation=format_evaluation(ev))},
    ]
    parsed, response = run_structured(messages, SummaryLLMOutput, temperature)
    return SummaryResult(**parsed.model_dump(),
                         InputTokens=response.usage.prompt_tokens,
                         OutputTokens=response.usage.completion_tokens)


summary_v2 = enhance_summary(document_text, summary_v1, eval_v1, TONE)
eval_v2 = evaluate_summary(document_text, summary_v2.Summary, TONE)

print(summary_v2.Summary, "\n")

comparison = pd.DataFrame({"v1 (original)": eval_v1.model_dump(), "v2 (enhanced)": eval_v2.model_dump()})
scores = comparison.loc[[i for i in comparison.index if i.endswith("Score")]].astype(float)
scores["change"] = scores["v2 (enhanced)"] - scores["v1 (original)"]
display(scores.round(2))
print("Output tokens  v1:", summary_v1.OutputTokens, "| v2:", summary_v2.OutputTokens)

Please, do not forget to add your comments.


# Submission Information

🚨 **Please review our [Assignment Submission Guide](https://github.com/UofT-DSI/onboarding/blob/main/onboarding_documents/submissions.md)** 🚨 for detailed instructions on how to format, branch, and submit your work. Following these guidelines is crucial for your submissions to be evaluated correctly.

## Submission Parameters

- The Submission Due Date is indicated in the [readme](../README.md#schedule) file.
- The branch name for your repo should be: assignment-1
- What to submit for this assignment:
    + This Jupyter Notebook (assignment_1.ipynb) should be populated and should be the only change in your pull request.
- What the pull request link should look like for this assignment: `https://github.com/<your_github_username>/production/pull/<pr_id>`
    + Open a private window in your browser. Copy and paste the link to your pull request into the address bar. Make sure you can see your pull request properly. This helps the technical facilitator and learning support staff review your submission easily.

## Checklist

+ Created a branch with the correct naming convention.
+ Ensured that the repository is public.
+ Reviewed the PR description guidelines and adhered to them.
+ Verify that the link is accessible in a private browser window.

If you encounter any difficulties or have questions, please don't hesitate to reach out to our team via our Slack. Our Technical Facilitators and Learning Support staff are here to help you navigate any challenges.
